# ⚾ KBO 기사 수집 데이터의 Snippet 추출 및 데이터베이스(SQLite) 연동 실습

본 노트북은 수집된 야구 뉴스 CSV 데이터(`아빌라_기사수집_260909_152613.csv`)를 바탕으로,
1. **CSV 데이터의 결측치(비어있는 Snippet) 상태 확인**
2. **기사 원문 URL로부터 기사 본문 요약(Snippet) 추출 및 보강**
3. **SQLite 데이터베이스 연결 및 `articles` 테이블 스키마 생성**
4. **보강된 데이터를 데이터베이스에 적재 (Insert or Ignore)**
5. **데이터베이스로부터 Snippet을 다각도로 조회 및 추출(SELECT)하는 SQL 쿼리 실습**
6. **추출한 Snippet 데이터를 AI 프롬프트 및 분석용으로 가공·활용하는 방법**

전 과정을 단계별로 다룹니다.

In [27]:
# 필요 라이브러리 임포트 (Python 기본 내장 sqlite3 사용 - 추가 설치 불필요)
import re
import time
import sqlite3
import requests
import pandas as pd
from bs4 import BeautifulSoup

print('라이브러리 로드 완료: pandas, sqlite3, requests, BeautifulSoup')

라이브러리 로드 완료: pandas, sqlite3, requests, BeautifulSoup


## 1. CSV 데이터 로드 및 Snippet 상태 점검
- 수집된 CSV 파일을 DataFrame으로 로드합니다.
- 현재 데이터에서 `snippet` 컬럼이 비어있는지(NaN) 확인합니다.

In [28]:
csv_path = '아빌라_기사수집_260909_152613.csv'
df = pd.read_csv(csv_path)

print(f'총 수집 기사 수: {len(df)}건')
print(f'데이터프레임 컬럼 목록: {list(df.columns)}')
print(f'snippet 컬럼 결측치(NaN) 수: {df["snippet"].isna().sum()}건 / 전체 {len(df)}건')

# 상위 5건 데이터 확인
df[['id', 'press', 'title', 'snippet']].head(5)

FileNotFoundError: [Errno 2] No such file or directory: '아빌라_기사수집_260909_152613.csv'

## 2. 기사 원문 URL로부터 본문 발췌(Snippet) 추출 및 데이터 보강
- CSV의 `snippet` 컬럼이 비어 있으므로, 각 행의 기사 `url`에 접속하여 본문 상위 150~200자를 추출하여 채워줍니다.
- **원칙 준수**: DataFrame의 컬럼 구성을 임의로 추가하거나 삭제하지 않고, 기존 `snippet` 컬럼의 결측치만 정제하여 채웁니다.
- **dtype 주의**: 전체 결측치인 컬럼은 pandas가 기본적으로 `float64`로 인식하므로, 문자열 삽입 전 `df['snippet'] = df['snippet'].astype(object)`로 타입을 변환합니다.

In [ ]:
def extract_snippet_from_url(url: str, max_chars: int = 180) -> str:
    """네이버 스포츠 기사 상세 페이지 본문에서 앞부분 max_chars자를 요약문(snippet)으로 추출"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    }
    try:
        resp = requests.get(url, headers=headers, timeout=6)
        if resp.status_code != 200:
            return ''
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        # 최신 네이버 스포츠 본문 컨테이너 또는 레거시 본문 영역 탐색
        content_div = (
            soup.find('div', class_='_article_content')
            or soup.find('article', class_='_article_body')
            or soup.find('div', id='newsEndContents')
            or soup.find('div', class_=lambda c: c and ('news_end' in c or 'artice_body' in c))
        )
        if content_div:
            # 본문 텍스트 div는 보존하고 스크립트, 스타일, iframe, 버튼 등만 제거
            for tag in content_div(['script', 'style', 'iframe', 'button']):
                tag.decompose()
            text = content_div.get_text(separator=' ').strip()
            # 연속된 공백 및 줄바꿈 정리
            text = re.sub(r'\s+', ' ', text)
            return text[:max_chars].strip()
        return ''
    except Exception:
        return ''

print('기사 본문으로부터 Snippet 추출 및 보강 시작...')
# 결측치(NaN)로 인해 float64로 자동 추론된 컬럼 타입을 문자열 저장이 가능한 object 타입으로 변환
df['snippet'] = df['snippet'].astype(object)

# 전체 16건의 기사 URL에서 본문 요약문 추출
for idx, row in df.iterrows():
    if pd.isna(row['snippet']) or str(row['snippet']).strip() == '':
        snippet_text = extract_snippet_from_url(row['url'], max_chars=180)
        df.at[idx, 'snippet'] = snippet_text
        time.sleep(0.1)  # 네이버 서버 부하 방지 딜레이

print('Snippet 보강 완료!')
print(f'보강 후 snippet 유효 건수: {df["snippet"].str.len().gt(0).sum()}건')
df[['press', 'title', 'snippet']].head(5)

기사 본문으로부터 Snippet 추출 및 보강 시작...
Snippet 보강 완료!
보강 후 snippet 유효 건수: 16건


,press,title,snippet
0,스포츠동아,"대체 외인 성공 사례 SSG 아빌라, 공포의 투심으로 경계 대상 1순위",SSG 페드로 아빌라는 최근 연이은 호투로 9개 구단 경계 대상 1순위로 떠올랐다....
1,스포츠조선,화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...,광주=박재만 기자 pjm@sportschosun.com 광주=박재만 기자 pjm@s...
2,스타뉴스,"'이럴수가' KBO 리그 9개 구단 초비상→'구단 최초 역사' 괴물 투수 ""한...",[스타뉴스 | 인천=김우종 기자] 4일 인천 SSG 랜더스 필드에서 펼쳐진 두산 베...
3,OSEN,"‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...","SSG 랜더스 제공 SSG 랜더스 제공 [OSEN=인천, 이후광 기자] 어디서 이런..."
4,OSEN,"韓 9개 구단 초비상! 폰세 능가 외인, 생애 첫 완봉승→재계약 전격 요...","SSG 랜더스 제공 [OSEN=인천, 이후광 기자] SSG 랜더스 를 제외한 프로야..."


## 3. SQLite 데이터베이스 생성 및 테이블 스키마 정의
- 경량 파일 기반 RDBMS인 **SQLite**를 사용합니다 (`baseball_news.db`).
- `articles` 테이블을 생성하며, 동일한 기사가 중복 적재되지 않도록 `url` 컬럼에 `UNIQUE` 제약조건을 부여합니다.
- 검색 속도 향상을 위해 날짜(`date`)와 언론사(`press`)에 인덱스를 생성합니다.

In [ ]:
db_name = 'baseball_news.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# 기사 테이블 스키마 정의 (DDL)
create_table_sql = """
CREATE TABLE IF NOT EXISTS articles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    category TEXT,
    title TEXT NOT NULL,
    press TEXT,
    date TEXT,
    url TEXT UNIQUE,
    snippet TEXT,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);
"""
cursor.execute(create_table_sql)
cursor.execute('CREATE INDEX IF NOT EXISTS idx_articles_date ON articles(date);')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_articles_press ON articles(press);')
conn.commit()

print(f"데이터베이스 파일 '{db_name}' 및 'articles' 테이블 스키마 생성 완료!")

데이터베이스 파일 'baseball_news.db' 및 'articles' 테이블 스키마 생성 완료!


## 4. 보강된 기사 데이터를 데이터베이스(DB)로 적재 (UPSERT)
- `ON CONFLICT(url) DO UPDATE` 구문(UPSERT)을 사용하여 이미 존재하는 기사 URL인 경우 비어있던 `snippet` 등의 정보를 최신 상태로 갱신하고, 새로운 기사는 신규 삽입합니다.
- 이를 통해 중복 수집 없이도 결측치 보강 작업이 안전하게 반영됩니다.

In [ ]:
insert_sql = """
INSERT INTO articles (category, title, press, date, url, snippet)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(url) DO UPDATE SET
    snippet = excluded.snippet,
    category = excluded.category,
    title = excluded.title,
    press = excluded.press,
    date = excluded.date;
"""

inserted_count = 0
for _, row in df.iterrows():
    cursor.execute(insert_sql, (
        row['category'],
        row['title'],
        row['press'],
        row['date'],
        row['url'],
        row['snippet']
    ))
    inserted_count += 1

conn.commit()
print(f"DB 적재 및 갱신 완료: 총 {inserted_count}건의 기사 처리 완료")

# DB에 실제로 저장된 총 행 수 및 snippet 유효 건수 확인
cursor.execute('SELECT COUNT(*), COUNT(snippet) FROM articles WHERE snippet IS NOT NULL AND snippet != ""')
valid_snippet_count = cursor.fetchone()[0]
cursor.execute('SELECT COUNT(*) FROM articles')
total_db_count = cursor.fetchone()[0]
print(f"현재 DB 내 전체 기사 수: {total_db_count}건 (유효 Snippet 보유: {valid_snippet_count}건)")

DB 적재 및 갱신 완료: 총 16건의 기사 처리 완료
현재 DB 내 전체 기사 수: 16건 (유효 Snippet 보유: 16건)


## 5. 데이터베이스에서 Snippet 데이터 추출하기 (핵심 쿼리)
데이터베이스에 저장된 데이터로부터 다양한 분석 및 응용 목적에 맞게 `snippet`을 SQL 쿼리로 추출합니다.

### 5-1. 기본 추출: Snippet이 비어있지 않은 기사 전체 조회

In [ ]:
query_all_snippets = """
SELECT id, press, title, snippet
FROM articles
WHERE snippet IS NOT NULL AND snippet != ''
ORDER BY id ASC
LIMIT 5;
"""

df_result_all = pd.read_sql_query(query_all_snippets, conn)
print('=== [추출 결과 1] 기본 Snippet 조회 (상위 5건) ===')
df_result_all

=== [추출 결과 1] 기본 Snippet 조회 (상위 5건) ===


,id,press,title,snippet
0,1,스포츠동아,"대체 외인 성공 사례 SSG 아빌라, 공포의 투심으로 경계 대상 1순위",SSG 페드로 아빌라는 최근 연이은 호투로 9개 구단 경계 대상 1순위로 떠올랐다....
1,2,스포츠조선,화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...,광주=박재만 기자 pjm@sportschosun.com 광주=박재만 기자 pjm@s...
2,3,스타뉴스,"'이럴수가' KBO 리그 9개 구단 초비상→'구단 최초 역사' 괴물 투수 ""한...",[스타뉴스 | 인천=김우종 기자] 4일 인천 SSG 랜더스 필드에서 펼쳐진 두산 베...
3,4,OSEN,"‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...","SSG 랜더스 제공 SSG 랜더스 제공 [OSEN=인천, 이후광 기자] 어디서 이런..."
4,5,OSEN,"韓 9개 구단 초비상! 폰세 능가 외인, 생애 첫 완봉승→재계약 전격 요...","SSG 랜더스 제공 [OSEN=인천, 이후광 기자] SSG 랜더스 를 제외한 프로야..."


### 5-2. 조건부 추출: Snippet 본문에 특정 키워드('완봉', '투심', '에이스')가 포함된 기사만 선별
- SQL의 `LIKE '%키워드%'` 조건을 통해 특정 단어가 언급된 snippet만 정밀하게 필터링하여 추출합니다.

In [ ]:
target_word = '완봉'
query_keyword = f"""
SELECT id, press, title, snippet
FROM articles
WHERE snippet LIKE '%{target_word}%'
"""

df_keyword = pd.read_sql_query(query_keyword, conn)
print(f"=== [추출 결과 2] Snippet 내 '{target_word}' 키워드 포함 기사: 총 {len(df_keyword)}건 ===")

for _, row in df_keyword.head(3).iterrows():
    print(f"[{row['press']}] {row['title']}")
    print(f"   발췌(Snippet): {row['snippet']}")
    print('-' * 80)

=== [추출 결과 2] Snippet 내 '완봉' 키워드 포함 기사: 총 11건 ===
[스포츠조선] 화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...
   발췌(Snippet): 광주=박재만 기자 pjm@sportschosun.com 광주=박재만 기자 pjm@sportschosun.com [스포츠조선 고재완 기자] 페드로 아빌라 (29)가 지난 4일 인천 두산 베어스 전에서 101구 3안타 완봉승을 수확하며 재계약의 확실한 도장을 찍은 반면, 대체 외인 토머스 해치(32)는 어깨 통증이 가라앉지
--------------------------------------------------------------------------------
[OSEN] ‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...
   발췌(Snippet): SSG 랜더스 제공 SSG 랜더스 제공 [OSEN=인천, 이후광 기자] 어디서 이런 괴물 외국인투수를 데려온 걸까. 연봉 5억 원을 받는 대체 외국인투수가 SSG 랜더스 창단 첫 완봉승의 주인공으로 우뚝 섰다. 프로야구 SSG 랜더스는 4일 인천SSG랜더스필드에서 열린 2026 신한 SOL KBO리그 두산 베어스와의 시즌
--------------------------------------------------------------------------------
[OSEN] 韓 9개 구단 초비상! 폰세 능가 외인, 생애 첫 완봉승→재계약 전격 요...
   발췌(Snippet): SSG 랜더스 제공 [OSEN=인천, 이후광 기자] SSG 랜더스 를 제외한 프로야구 9개 구단에 비상이 걸렸다. 지난해 정규시즌 MVP 코디 폰세(토론토 블루제이스)를 능가하는 괴물 외국인투수가 생애 첫 완봉승을 달성한 뒤 앞으로 2년 더 SSG에서 뛰고 싶다는 뜻을 밝혔다. SSG 외국인투수 페드로 아빌라 는 지난 4
-------------------------------------------------

### 5-3. 집계 및 통계 추출: 언론사별 Snippet 수 및 평균 글자 수

In [ ]:
query_stats = """
SELECT 
    press,
    COUNT(*) AS article_count,
    ROUND(AVG(LENGTH(snippet)), 1) AS avg_snippet_length
FROM articles
WHERE snippet IS NOT NULL AND snippet != ''
GROUP BY press
ORDER BY article_count DESC;
"""

df_stats = pd.read_sql_query(query_stats, conn)
print('=== [추출 결과 3] 언론사별 Snippet 집계 현황 ===')
df_stats.head(10)

=== [추출 결과 3] 언론사별 Snippet 집계 현황 ===


,press,article_count,avg_snippet_length
0,OSEN,2,180.0
1,연합뉴스,2,174.0
2,뉴스1,1,180.0
3,뉴시스,1,180.0
4,마이데일리,1,180.0
5,스타뉴스,1,180.0
6,스포츠경향,1,179.0
7,스포츠동아,1,180.0
8,스포츠조선,1,179.0
9,엑스포츠뉴스,1,180.0


## 6. 추출된 Snippet 데이터의 실전 활용 (AI 프롬프트/분석 코퍼스 변환)
- DB에서 추출한 `snippet` 데이터를 리스트나 딕셔너리로 순회하여, AI 요약 보고서 작성용 입력 텍스트 포맷으로 가공합니다.

In [ ]:
# DB에서 전체 snippet 목록 추출
cursor.execute('SELECT date, press, title, snippet FROM articles WHERE snippet IS NOT NULL AND snippet != ""')
article_records = cursor.fetchall()

# AI 분석 보고서에 입력할 구조화된 텍스트로 결합
briefing_snippets = []
for date, press, title, snip in article_records[:5]:
    briefing_snippets.append(f"- [{date} | {press}] {title}\n  ▶ 요약 발췌: {snip}")

final_corpus = "\n\n".join(briefing_snippets)
print('=== [실전 활용] DB에서 추출된 Snippet 기반 AI 분석용 프롬프트 데이터 ===\n')
print(final_corpus)

# 작업 완료 후 데이터베이스 연결 닫기
conn.close()
print('\n데이터베이스 연결이 안전하게 종료되었습니다.')

=== [실전 활용] DB에서 추출된 Snippet 기반 AI 분석용 프롬프트 데이터 ===

- [2026-09-09 | 스포츠동아] 대체 외인 성공 사례 SSG 아빌라, 공포의 투심으로 경계 대상 1순위
  ▶ 요약 발췌: SSG 페드로 아빌라는 최근 연이은 호투로 9개 구단 경계 대상 1순위로 떠올랐다. 가을야구 순위 싸움이 치열한 팀들로서는 아빌라와 맞대결이 유독 더 신경 쓰일 수밖에 없다. 사진제공｜SSG 랜더스 [스포츠동아 장은상 기자] SSG 랜더스 외국인 투수 페드로 아빌라 (29)가 경계 대상 1순위로 급부상했다. 아빌라는 SS

- [2026-09-09 | 스포츠조선] 화이트 퇴출→베니지아노 방출→해치 시즌아웃…外人 잔혹사 끊어낸 아...
  ▶ 요약 발췌: 광주=박재만 기자 pjm@sportschosun.com 광주=박재만 기자 pjm@sportschosun.com [스포츠조선 고재완 기자] 페드로 아빌라 (29)가 지난 4일 인천 두산 베어스 전에서 101구 3안타 완봉승을 수확하며 재계약의 확실한 도장을 찍은 반면, 대체 외인 토머스 해치(32)는 어깨 통증이 가라앉지

- [2026-09-09 | 스타뉴스] '이럴수가' KBO 리그 9개 구단 초비상→'구단 최초 역사' 괴물 투수 "한...
  ▶ 요약 발췌: [스타뉴스 | 인천=김우종 기자] 4일 인천 SSG 랜더스 필드에서 펼쳐진 두산 베어스와 홈 경기에서 SSG 랜더스 외국인 투수 페드로 아빌라의 활약 모습. /사진=SSG 랜더스 제공 4일 인천 SSG 랜더스 필드에서 펼쳐진 두산 베어스와 홈 경기에서 SSG 랜더스 외국인 투수 페드로 아빌라의 활약 모습. 경기 후 취재진

- [2026-09-09 | OSEN] ‘연봉 고작 5억’ 어디서 이런 괴물을 데려왔나, 창단 첫 완봉승→사령...
  ▶ 요약 발췌: SSG 랜더스 제공 SSG 랜더스 제공 [OSEN=인천, 이후광 기자] 어디서 이런 괴물 외국인투수를 데려온 걸까. 연봉 5억 원을 받는 대체 외국인투수가 SSG 랜더스 창단 

## 7. 키워드 정규화(Keyword Normalization) 및 동의어 통합 시스템 구현

동일한 대상을 지칭하지만 키워드 표기 차이(예: `한화` vs `한화이글스`, `KIA` vs `기아`)로 인해 발생하는
1. **DB 파일 분산 저장(파편화)** 현상 방지
2. **AI 보고서 핵심 기사 연관도 산출(`get_relevance_score`) 누락** 방지
3. **챗봇 RAG 검색 정합성 저하** 방지

를 위해 **KBO 도메인 사전 규칙**과 **OpenAI `gpt-5.6-luna` 지능형 Fallback**을 결합한 키워드 정규화 모듈을 구현하고 검증합니다.

In [ ]:
# ============================================================
# 1. 키워드 정규화 엔진 (KeywordNormalizer) 구현
# ============================================================
import os
import re
import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

load_dotenv(override=True)

@dataclass
class NormalizedEntity:
    raw: str                  # 사용자가 입력한 원본 문자열
    canonical: str            # 화면 및 보고서 제목용 대표 공식 명칭 (예: '한화 이글스')
    safe_id: str              # DB 파일명 및 시스템 식별자용 명칭 (예: '한화이글스')
    synonyms: List[str]       # 동의어 및 축약어 풀 (연관도 검색 및 RAG 매칭용)
    entity_type: str          # 'team'(구단), 'player'(선수), 'topic'(야구주제), 'general'(일반)

class KeywordNormalizer:
    """
    KBO 야구 뉴스 수집, AI 보고서, DB 적재를 위한 하이브리드 키워드 정규화기
    - 1차: KBO 10개 구단 및 야구 용어 사전 기반 고속 매핑 (Zero Latency)
    - 2차: 텍스트 전처리 (선수명 띄어쓰기 결합, 불필요한 조사 제거)
    - 3차: OpenAI gpt-5.6-luna 지능형 Fallback & 캐싱
    """

    # KBO 10개 구단 공식 명칭 및 동의어 매핑 테이블
    KBO_TEAMS = {
        "한화이글스": {
            "canonical": "한화 이글스",
            "safe_id": "한화이글스",
            "synonyms": ["한화", "한화이글스", "한화 이글스", "이글스"],
            "entity_type": "team",
        },
        "KIA타이거즈": {
            "canonical": "KIA 타이거즈",
            "safe_id": "KIA타이거즈",
            "synonyms": ["KIA", "기아", "기아타이거즈", "KIA타이거즈", "기아 타이거즈", "KIA 타이거즈", "타이거즈"],
            "entity_type": "team",
        },
        "LG트윈스": {
            "canonical": "LG 트윈스",
            "safe_id": "LG트윈스",
            "synonyms": ["LG", "엘지", "LG트윈스", "엘지트윈스", "LG 트윈스", "엘지 트윈스", "트윈스"],
            "entity_type": "team",
        },
        "SSG랜더스": {
            "canonical": "SSG 랜더스",
            "safe_id": "SSG랜더스",
            "synonyms": ["SSG", "쓱", "SSG랜더스", "SSG 랜더스", "랜더스", "SK와이번스", "SK 와이번스"],
            "entity_type": "team",
        },
        "두산베어스": {
            "canonical": "두산 베어스",
            "safe_id": "두산베어스",
            "synonyms": ["두산", "두산베어스", "두산 베어스", "베어스"],
            "entity_type": "team",
        },
        "삼성라이온즈": {
            "canonical": "삼성 라이온즈",
            "safe_id": "삼성라이온즈",
            "synonyms": ["삼성", "삼성라이온즈", "삼성 라이온즈", "라이온즈"],
            "entity_type": "team",
        },
        "롯데자이언츠": {
            "canonical": "롯데 자이언츠",
            "safe_id": "롯데자이언츠",
            "synonyms": ["롯데", "롯데자이언츠", "롯데 자이언츠", "자이언츠"],
            "entity_type": "team",
        },
        "KT위즈": {
            "canonical": "KT 위즈",
            "safe_id": "KT위즈",
            "synonyms": ["KT", "케이티", "KT위즈", "케이티위즈", "KT 위즈", "위즈"],
            "entity_type": "team",
        },
        "NC다이노스": {
            "canonical": "NC 다이노스",
            "safe_id": "NC다이노스",
            "synonyms": ["NC", "엔씨", "NC다이노스", "엔씨다이노스", "NC 다이노스", "다이노스"],
            "entity_type": "team",
        },
        "키움히어로즈": {
            "canonical": "키움 히어로즈",
            "safe_id": "키움히어로즈",
            "synonyms": ["키움", "키움히어로즈", "키움 히어로즈", "히어로즈", "넥센", "넥센히어로즈"],
            "entity_type": "team",
        },
    }

    # 주요 야구 일반 토픽 사전
    KBO_TOPICS = {
        "가을야구": ["포스트시즌", "가을야구", "PS", "준플레이오프", "플레이오프", "한국시리즈"],
        "FA": ["FA", "자유계약", "자유계약선수", "프리에이전트"],
        "신인드래프트": ["드래프트", "신인드래프트", "신인선수지명"],
    }

    def __init__(self, client: Optional[OpenAI] = None, model: str = "gpt-5.6-luna"):
        api_key = os.getenv("OPENAI_API_KEY")
        self.client = client or (OpenAI(api_key=api_key) if api_key else None)
        self.model = model
        self._cache: Dict[str, NormalizedEntity] = {}

        # 빠른 역방향 매핑 색인 (소문자/공백제거 -> 대상 키)
        self._lookup: Dict[str, str] = {}
        for team_key, info in self.KBO_TEAMS.items():
            for syn in info["synonyms"]:
                clean_syn = self._clean_token(syn)
                self._lookup[clean_syn] = team_key

    @staticmethod
    def _clean_token(text: str) -> str:
        return re.sub(r"\s+", "", text.lower())

    def _strip_josa(self, text: str) -> str:
        """불필요한 한국어 조사(은/는/이/가/의/을/를/과/와/에게/에서/도) 제거"""
        josa_pattern = r"(은|는|이|가|의|을|를|과|와|에게|에서|도|에)$"
        if len(text) > 2:
            return re.sub(josa_pattern, "", text).strip()
        return text.strip()

    def normalize(self, keyword: str, use_llm_fallback: bool = True) -> NormalizedEntity:
        """
        입력 키워드를 표준 엔티티(NormalizedEntity)로 변환합니다.
        """
        raw = keyword.strip()
        if not raw:
            return NormalizedEntity(raw="", canonical="야구", safe_id="야구", synonyms=["야구"], entity_type="general")

        # 캐시 확인
        if raw in self._cache:
            return self._cache[raw]

        # 1. 텍스트 기본 정제 및 조사 제거
        cleaned = self._strip_josa(raw)
        lookup_key = self._clean_token(cleaned)

        # 2. KBO 10개 구단 사전 검사
        if lookup_key in self._lookup:
            team_key = self._lookup[lookup_key]
            info = self.KBO_TEAMS[team_key]
            entity = NormalizedEntity(
                raw=raw,
                canonical=info["canonical"],
                safe_id=info["safe_id"],
                synonyms=list(set(info["synonyms"] + [raw, cleaned])),
                entity_type=info["entity_type"],
            )
            self._cache[raw] = entity
            return entity

        # 3. KBO 주요 토픽 사전 검사
        for topic_name, syn_list in self.KBO_TOPICS.items():
            for s in syn_list:
                if self._clean_token(s) == lookup_key:
                    entity = NormalizedEntity(
                        raw=raw,
                        canonical=topic_name,
                        safe_id=topic_name,
                        synonyms=list(set(syn_list + [raw, cleaned])),
                        entity_type="topic",
                    )
                    self._cache[raw] = entity
                    return entity

        # 4. 한국어 인명/선수명 형태 감지 (예: '김 도 영' -> '김도영')
        if re.fullmatch(r"^[가-힣]\s+[가-힣](\s+[가-힣])?$", cleaned):
            combined_name = re.sub(r"\s+", "", cleaned)
            entity = NormalizedEntity(
                raw=raw,
                canonical=combined_name,
                safe_id=combined_name,
                synonyms=list(set([raw, cleaned, combined_name])),
                entity_type="player",
            )
            self._cache[raw] = entity
            return entity

        # 5. LLM Fallback (OpenAI gpt-5.6-luna) - 사전에 없는 별명, 오타, 복합 표현 보정
        if use_llm_fallback and self.client:
            try:
                prompt = f"""당신은 한국 프로야구(KBO) 전문 데이터 엔지니어입니다.
사용자가 입력한 검색 키워드를 분석하여 표준 대표 명칭과 동의어를 JSON으로 정규화하세요.

입력 키워드: "{raw}"

지침:
1. canonical: 공식 표준 명칭 (구단이면 정식명칭, 선수면 선수명, 주제면 표준용어)
2. safe_id: 공백 및 특수문자가 없는 파일/DB용 식별자
3. synonyms: 기사 검색 및 연관도 산출에 사용할 동의어/약칭/별칭 리스트 (최대 5개)
4. entity_type: "team", "player", "topic", "general" 중 하나

반드시 순수 JSON 형식만 한 줄로 출력하세요:
{{"canonical": "...", "safe_id": "...", "synonyms": [...], "entity_type": "..."}}"""
                resp = self.client.responses.create(
                    model=self.model,
                    instructions="You are a strict KBO baseball entity normalizer. Output pure JSON only without markdown.",
                    input=prompt,
                )
                text = resp.output_text.strip()
                if text.startswith("```"):
                    text = text.split("```")[1]
                    if text.startswith("json"):
                        text = text[4:]
                data = json.loads(text.strip())

                canonical = data.get("canonical", cleaned)
                safe_id = re.sub(r"[^\w가-힣0-9_-]", "", data.get("safe_id", canonical)).strip() or cleaned
                syns = list(set([raw, cleaned, canonical] + data.get("synonyms", [])))
                etype = data.get("entity_type", "general")

                entity = NormalizedEntity(raw=raw, canonical=canonical, safe_id=safe_id, synonyms=syns, entity_type=etype)
                self._cache[raw] = entity
                return entity
            except Exception:
                pass

        # Fallback 기본값: 정제된 단어 사용
        safe_clean = re.sub(r"[^\w가-힣0-9_-]", "", cleaned).strip() or "야구"
        entity = NormalizedEntity(
            raw=raw,
            canonical=cleaned,
            safe_id=safe_clean,
            synonyms=list(set([raw, cleaned])),
            entity_type="general",
        )
        self._cache[raw] = entity
        return entity

print('KeywordNormalizer 클래스 정의 완료!')

KeywordNormalizer 클래스 정의 완료!


### 7.1. 다양한 형태의 키워드 정규화 테스트
- 구단 약칭, 띄어쓰기 오타, 선수명 조사 결합 등 실제 사용자 입력 패턴을 넣어 정규화 결과를 확인합니다.

In [33]:
# 정규화 엔진 인스턴스 생성
normalizer = KeywordNormalizer(model="gpt-5.6-luna")

# 테스트할 키워드 목록 (다양한 표기 변형)
test_keywords = [
    "한화",
    "한화이글스",
    "한화 이글스",
    "이글스",
    "기아",
    "KIA",
    "KIA 타이거즈",
    "엘지",
    "LG 트윈스",
    "쓱",
    "SSG 랜더스",
    "엔씨",
    "김 도 영",
    "김도영의",
    "가을야구",
]

results = []
for kw in test_keywords:
    res = normalizer.normalize(kw, use_llm_fallback=False)  # 1차 사전/규칙 테스트
    results.append({
        "입력 키워드": res.raw,
        "표준 공식명 (canonical)": res.canonical,
        "DB/파일명 ID (safe_id)": res.safe_id,
        "분류": res.entity_type,
        "동의어 풀 (synonyms)": ", ".join(res.synonyms),
    })

df_norm = pd.DataFrame(results)
print("=== [정규화 엔진 테스트 결과] ===")
df_norm

NameError: name 'KeywordNormalizer' is not defined

### 7.2. 보고서 핵심 기사 발췌 시 연관도 점수(`get_relevance_score`) 비교 실습
- **기존 방식**: 사용자가 '한화 이글스'로 검색했을 때, 기사 제목/본문에 '한화'라고만 적혀 있으면 점수가 0점이 되어 핵심 기사에서 탈락하는 문제 발생
- **신규 방식**: 정규화된 `synonyms` 풀 전체를 대조하여 모든 약칭/동의어 출현 빈도를 점수에 합산

In [ ]:
# 가상의 기사 샘플 데이터
sample_articles = [
    {
        "title": "한화, 3연승 질주... 불펜 완벽 계투로 승리 견인",
        "content": "대전 경기에서 한화는 8회 터진 결승타로 극적인 승리를 거두었다. 이글스 팬들은 열광했다."
    },
    {
        "title": "KIA 타이거즈 김도영, 시즌 30호 홈런 작렬 대기록",
        "content": "광주 경기에서 기아의 중심 타자 김도영이 호쾌한 타격으로 승리를 이끌었다."
    },
    {
        "title": "LG 트윈스 가을야구 청신호, 잠실서 두산 격파",
        "content": "엘지 트윈스가 완벽한 투타 조화로 두산 베어스를 꺾고 상위권을 수성했다."
    }
]

def old_relevance_score(art: dict, keyword: str) -> int:
    """기존 단일 문자열 일치 방식"""
    t = art.get("title", "")
    c = art.get("content", "")
    return (t.count(keyword) * 15) + (c.count(keyword) * 3)

def new_relevance_score(art: dict, entity: NormalizedEntity) -> int:
    """신규 동의어 풀(Synonyms) 가중치 합산 방식"""
    t = art.get("title", "")
    c = art.get("content", "")
    score = 0
    # 중복 가산을 방지하기 위해 각 동의어의 출현 빈도를 합산
    for syn in entity.synonyms:
        if not syn:
            continue
        score += t.count(syn) * 15
        score += c.count(syn) * 3
    return score

# 사용자가 '한화 이글스'로 검색한 시나리오 테스트
search_query = "한화 이글스"
entity = normalizer.normalize(search_query)

print(f"[테스트 쿼리]: '{search_query}' (정규화 표준: '{entity.canonical}', 동의어: {entity.synonyms})\n")

comparison = []
for idx, art in enumerate(sample_articles, 1):
    old_score = old_relevance_score(art, search_query)
    new_score = new_relevance_score(art, entity)
    comparison.append({
        "기사 제목": art["title"],
        "기존 점수 (단일 일치)": old_score,
        "신규 점수 (동의어 풀)": new_score,
        "판정": "정상 발췌 (개선)" if old_score == 0 and new_score > 0 else "일치"
    })

df_comp = pd.DataFrame(comparison)
df_comp

[테스트 쿼리]: '한화 이글스' (정규화 표준: '한화 이글스', 동의어: ['한화', '한화이글스', '이글스', '한화 이글스'])



,기사 제목,기존 점수 (단일 일치),신규 점수 (동의어 풀),판정
0,"한화, 3연승 질주... 불펜 완벽 계투로 승리 견인",0,21,정상 발췌 (개선)
1,"KIA 타이거즈 김도영, 시즌 30호 홈런 작렬 대기록",0,0,일치
2,"LG 트윈스 가을야구 청신호, 잠실서 두산 격파",0,0,일치


### 7.3. DB 파일명 일원화 및 정규화 데이터 적재 시뮬레이션
- 사용자가 `한화`, `한화이글스`, `이글스` 중 어떤 것으로 수집하더라도 `safe_id`를 기반으로 `한화이글스_데이터베이스.db` 단일 파일에 누적 적재되는 로직을 검증합니다.

In [ ]:
# DB 파일명 결정 로직 시뮬레이션
user_inputs = ["한화", "한화이글스", "한화 이글스", "기아", "KIA", "KIA 타이거즈"]

db_mapping_results = []
for u_in in user_inputs:
    ent = normalizer.normalize(u_in)
    target_db_filename = f"{ent.safe_id}_데이터베이스.db"
    target_report_filename = f"{ent.safe_id}_보고서_260911.md"
    
    db_mapping_results.append({
        "사용자 입력 키워드": u_in,
        "정규화 대표 공식명": ent.canonical,
        "일원화된 DB 파일명": target_db_filename,
        "일원화된 리포트 파일명": target_report_filename,
    })

df_db_map = pd.DataFrame(db_mapping_results)
print("=== [DB 및 리포트 파일명 일원화 매핑 결과] ===")
print("-> '한화', '한화이글스', '한화 이글스' 모두 동일한 단일 DB 파일로 매핑되어 누적됩니다.")
df_db_map

=== [DB 및 리포트 파일명 일원화 매핑 결과] ===
-> '한화', '한화이글스', '한화 이글스' 모두 동일한 단일 DB 파일로 매핑되어 누적됩니다.


,사용자 입력 키워드,정규화 대표 공식명,일원화된 DB 파일명,일원화된 리포트 파일명
0,한화,한화 이글스,한화이글스_데이터베이스.db,한화이글스_보고서_260911.md
1,한화이글스,한화 이글스,한화이글스_데이터베이스.db,한화이글스_보고서_260911.md
2,한화 이글스,한화 이글스,한화이글스_데이터베이스.db,한화이글스_보고서_260911.md
3,기아,KIA 타이거즈,KIA타이거즈_데이터베이스.db,KIA타이거즈_보고서_260911.md
4,KIA,KIA 타이거즈,KIA타이거즈_데이터베이스.db,KIA타이거즈_보고서_260911.md
5,KIA 타이거즈,KIA 타이거즈,KIA타이거즈_데이터베이스.db,KIA타이거즈_보고서_260911.md


---
# 🤖 [실습] SQLAlchemy 2.0 기반 AI 자연어 DB 제어 (DDL/CRUD/통계) 실습

본 섹션에서는 향후 서버 백엔드 확장(FastAPI/PostgreSQL 등)을 대비하여,
**SQLAlchemy 2.0 표준(`create_engine`, `inspect`, `text()`, Context Manager 트랜잭션)**과 **OpenAI `gpt-5.6-luna`**를 연동하여,
사용자가 자연어로 명령하면 AI가 최적의 SQLite SQL을 작성하고 데이터베이스에 안전하게 실행하는 파이프라인을 검증합니다.

### 🎯 주요 실습 단계
1. **SQLAlchemy 2.0 엔진 초기화 및 커넥션 풀 연결**
2. **`inspect(engine)`를 통한 표준 스키마 리플렉션 (메타데이터 자동 추출)**
3. **OpenAI `gpt-5.6-luna` 기반 자연어 ➔ SQLite SQL 작성 및 정제 (`text()`)**
4. **SELECT 통계 조회 테스트 (`pd.read_sql_query`)**
5. **DDL 테이블 생성 테스트 (`CREATE TABLE IF NOT EXISTS`)**
6. **CRUD 데이터 조작 테스트 (`INSERT`, `UPDATE`, `SELECT`)**
7. **오류 발생 시 AI 자가 수정(Self-Correction) 복구 테스트**

In [ ]:
# 1. 필요한 라이브러리 임포트 및 SQLAlchemy 2.0 엔진 초기화
import os
import re
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.exc import SQLAlchemyError
from openai import OpenAI

# 프로젝트 루트 및 .env 로드
PROJECT_ROOT = Path(".").resolve()
load_dotenv(PROJECT_ROOT / ".env", override=True)

# OpenAI 클라이언트 초기화 (기본 모델: gpt-5.6-luna)
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) if api_key else None
MODEL_NAME = "gpt-5.6-luna"

# 통합 데이터베이스 경로 지정
DB_PATH = PROJECT_ROOT / "storage" / "db" / "baseball_news.db"
db_url = f"sqlite:///{DB_PATH.resolve()}"

# SQLAlchemy 2.0 엔진 생성
engine = create_engine(db_url, echo=False)
print(f"✅ SQLAlchemy 2.0 엔진 생성 완료: {db_url}")
print(f"✅ OpenAI 클라이언트 연결 상태: {bool(client)} (모델: {MODEL_NAME})")

✅ SQLAlchemy 2.0 엔진 생성 완료: sqlite:///C:\Projects\mini-project_0909\storage\db\baseball_news.db
✅ OpenAI 클라이언트 연결 상태: True (모델: gpt-5.6-luna)


In [ ]:
# 2. inspect(engine)를 통한 표준 스키마 자동 추출 함수 정의
def get_schema_info(engine) -> str:
    """SQLAlchemy 2.0 inspect를 사용하여 현재 DB의 모든 테이블 및 컬럼, 레코드 수를 텍스트로 추출합니다."""
    inspector = inspect(engine)
    tables = inspector.get_table_names()
    if not tables:
        return "[현재 데이터베이스에 생성된 테이블이 없습니다.]"
    
    schema_parts = []
    with engine.connect() as conn:
        for table_name in tables:
            columns = inspector.get_columns(table_name)
            pk_constraint = inspector.get_pk_constraint(table_name)
            pk_cols = pk_constraint.get("constrained_columns", []) if pk_constraint else []
            
            col_desc = []
            for col in columns:
                c_name = col["name"]
                c_type = str(col["type"])
                pk_str = " (PRIMARY KEY)" if c_name in pk_cols else ""
                null_str = " NOT NULL" if not col.get("nullable", True) else ""
                col_desc.append(f"  - {c_name} ({c_type}){pk_str}{null_str}")
            
            try:
                cnt_res = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}"))
                count = cnt_res.scalar()
            except Exception:
                count = 0
            
            t_text = f"테이블명: {table_name} (현재 레코드: {count}건)\n컬럼 목록:\n" + "\n".join(col_desc)
            schema_parts.append(t_text)
    
    return "\n\n" + ("=" * 45) + "\n" + "\n\n".join(schema_parts) + "\n" + ("=" * 45)

# 스키마 추출 확인
schema_text = get_schema_info(engine)
print("[inspect(engine)로 추출된 실시간 DB 스키마]")
print(schema_text)

[inspect(engine)로 추출된 실시간 DB 스키마]


테이블명: ai_reports (현재 레코드: 9건)
컬럼 목록:
  - id (INTEGER) (PRIMARY KEY)
  - query_id (INTEGER)
  - report_md (TEXT)
  - created_at (DATETIME)

테이블명: article_keywords (현재 레코드: 551건)
컬럼 목록:
  - query_id (INTEGER) (PRIMARY KEY) NOT NULL
  - article_id (INTEGER) (PRIMARY KEY) NOT NULL

테이블명: articles (현재 레코드: 394건)
컬럼 목록:
  - id (INTEGER) (PRIMARY KEY)
  - category (TEXT)
  - title (TEXT) NOT NULL
  - press (TEXT)
  - date (TEXT)
  - url (TEXT) NOT NULL
  - content (TEXT)
  - created_at (DATETIME)

테이블명: search_queries (현재 레코드: 9건)
컬럼 목록:
  - id (INTEGER) (PRIMARY KEY)
  - keyword (TEXT) NOT NULL
  - start_date (TEXT)
  - end_date (TEXT)
  - collected_count (INTEGER) NOT NULL
  - executed_at (DATETIME)
  - canonical_keyword (TEXT)


In [ ]:
# 3. SQL 정제, 안전성 검증 및 쿼리 분류 헬퍼 함수
def clean_sql(raw_response: str) -> str:
    """AI 응답 텍스트에서 마크다운 및 불필요한 설명을 제거하고 순수 SQL문만 추출합니다."""
    text_content = raw_response.strip()
    match = re.search(r"```(?:sql)?\s*([\s\S]*?)\s*```", text_content, re.IGNORECASE)
    if match:
        sql = match.group(1).strip()
    else:
        kw_match = re.search(r"(SELECT|INSERT|UPDATE|DELETE|CREATE|DROP|ALTER|PRAGMA|WITH)\b", text_content, re.IGNORECASE)
        sql = text_content[kw_match.start():].strip() if kw_match else text_content
    
    sql = sql.strip("`'\"\n\r\t ")
    if sql and not sql.endswith(";"):
        sql += ";"
    return sql

def get_query_type(sql: str) -> str:
    """SQL 쿼리의 성격을 판별합니다 (READ / WRITE / DDL)."""
    cleaned = re.sub(r"--.*?\n", "", sql)
    cleaned = re.sub(r"/\*.*?\*/", "", cleaned, flags=re.DOTALL).strip()
    first_word = cleaned.split()[0].upper() if cleaned.split() else ""
    if first_word in ("SELECT", "PRAGMA", "WITH", "EXPLAIN"):
        return "READ"
    elif first_word in ("INSERT", "UPDATE", "DELETE"):
        return "WRITE"
    elif first_word in ("CREATE", "ALTER", "DROP"):
        return "DDL"
    return "UNKNOWN"

def check_query_safety(sql: str) -> tuple[bool, str]:
    """위험한 파괴적 쿼리 여부를 검증합니다."""
    upper = sql.upper()
    if "DROP TABLE" in upper:
        return False, "테이블을 완전히 삭제하는 DROP TABLE 구문은 실행할 수 없습니다."
    if re.search(r"\bDELETE\s+FROM\s+\w+\s*(?:;)?$", upper):
        return False, "WHERE 조건절이 없는 전체 행 DELETE 구문은 실행할 수 없습니다."
    return True, "안전함"

In [ ]:
# 4. 자연어 ➔ AI SQL 생성 및 SQLAlchemy 2.0 실행 함수 (대화 컨텍스트 메모리 & 자가 수정 포함)
_last_notebook_context = None  # 직전 대화 및 쿼리 실행 결과 컨텍스트 보관용

def ask_ai_db(user_query: str, engine, client, model: str = "gpt-5.6-luna", auto_correct: bool = True):
    """자연어 질의를 받아 SQLAlchemy 엔진 위에서 SQL을 생성하고 실행합니다. (직전 질의 결과 기억)"""
    global _last_notebook_context
    if not client:
        raise ValueError("OpenAI API 키가 설정되지 않았습니다.")
    
    schema_info = get_schema_info(engine)
    
    context_str = ""
    if _last_notebook_context:
        ctx = _last_notebook_context
        context_str = (
            f"\n[직전 대화 및 쿼리 실행 컨텍스트 (연속 작업용 참조 정보)]\n"
            f"- 직전 사용자 요청: \"{ctx.get('user_query', '')}\"\n"
            f"- 직전 실행 SQL: \"{ctx.get('sql', '')}\"\n"
            f"- 직전 대상 테이블: {ctx.get('table', 'articles')}\n"
            f"- 직전 조회 건수: {ctx.get('row_count', 0)}건\n"
            f"- 직전 조회된 레코드 ID 목록: {ctx.get('target_ids', [])}\n"
        )
    
    system_instructions = (
        "당신은 SQLAlchemy 및 SQLite 전문 데이터 엔지니어입니다.\n"
        "아래 데이터베이스 스키마와 사용자의 요청을 바탕으로, 실행 가능한 최적의 단일 SQLite SQL문만 작성하세요.\n\n"
        "[규칙]\n"
        "1. 부가 설명, 인사말 없이 오직 순수한 SQL문만 출력하세요.\n"
        "2. 스키마에 존재하는 테이블과 컬럼만 정확히 참조하세요.\n"
        "3. 특정 문구가 제목(title)에 포함된 기사 조건부 조회/삭제 시 LIKE '%문구%' 구문과 WHERE 절을 사용하세요.\n"
        "4. [연속 작업 지원]: 사용자가 '방금 조회된 기사들', '직전 결과' 등 이전 맥락을 지칭하는 경우 "
        "반드시 [직전 대화 및 쿼리 실행 컨텍스트]의 레코드 ID 목록(WHERE id IN (...)) 또는 조건을 활용하여 쿼리를 작성하세요.\n"
        "5. 테이블 생성(DDL) 시에는 CREATE TABLE IF NOT EXISTS 구문을 사용하세요."
    )
    
    prompt = f"""[현재 DB 스키마]
{schema_info}
{context_str}
[사용자 요청]
{user_query}
"""
    
    print(f"\n[🗣️ 사용자 질문] {user_query}")
    resp = client.responses.create(
        model=model,
        instructions=system_instructions,
        input=prompt
    )
    sql = clean_sql(resp.output_text)
    print(f"[🤖 AI 작성 SQL]\n{sql}")
    
    is_safe, warn = check_query_safety(sql)
    if not is_safe:
        print(f"⚠️ 쿼리 안전성 검사 차단: {warn}")
        return {"status": "blocked", "reason": warn, "sql": sql}
    
    q_type = get_query_type(sql)
    
    def execute_core(target_sql):
        global _last_notebook_context
        q_t = get_query_type(target_sql)
        t_match = re.search(r"\b(?:FROM|INTO|UPDATE)\s+([a-zA-Z0-9_]+)", target_sql, re.IGNORECASE)
        target_table = t_match.group(1) if t_match else "articles"
        
        if q_t == "READ":
            with engine.connect() as conn:
                df = pd.read_sql_query(text(target_sql), conn)
            records = df.to_dict(orient="records")
            target_ids = [r["id"] for r in records if "id" in r]
            _last_notebook_context = {
                "user_query": user_query,
                "sql": target_sql,
                "query_type": "READ",
                "table": target_table,
                "row_count": len(df),
                "target_ids": target_ids,
            }
            print(f"✅ 조회 성공: 총 {len(df)}건 반환 (ID: {target_ids})")
            return df
        else:
            with engine.begin() as conn:
                res = conn.execute(text(target_sql))
                affected = res.rowcount
            if q_t == "DELETE":
                _last_notebook_context = None
            msg = f"✅ {q_t} 실행 완료 (영향받은 행: {affected}건)"
            print(msg)
            return {"status": "success", "type": q_t, "affected": affected, "sql": target_sql}
    
    try:
        return execute_core(sql)
    except SQLAlchemyError as err:
        if not auto_correct:
            raise err
        
        print(f"\n[⚠️ 실행 오류 발생] {err}")
        print("🔄 AI에게 오류 메시지를 전달하여 자가 수정(Self-Correction)을 수행합니다...")
        fix_prompt = f"""[자가 수정 요청]
이전 시도했던 쿼리에서 오류가 발생했습니다.
- 실패한 SQL: {sql}
- 에러 내용: {str(err)}
- 원본 사용자 요청: {user_query}

위 에러의 원인을 파악하여 올바르게 작동하는 단일 SQLite SQL문만 다시 작성하세요.
"""
        fix_resp = client.responses.create(
            model=model,
            instructions="You are a strict SQL bug fixer. Output single corrected SQLite SQL only.",
            input=fix_prompt
        )
        fixed_sql = clean_sql(fix_resp.output_text)
        print(f"[🛠️ AI 보정 SQL]\n{fixed_sql}")
        return execute_core(fixed_sql)


### 📊 테스트 1: 자연어 데이터 통계 및 집계 조회 (READ / SELECT)
- 사용자의 자연어 요청: `"언론사별 기사 건수가 많은 순서대로 상위 5개 뽑아줘"`
- AI가 SQL 작성 ➔ `pd.read_sql_query(text(sql), conn)` 실행 ➔ DataFrame 표 출력

In [ ]:
# 테스트 1 실행
result_df1 = ask_ai_db("언론사별 기사 건수가 많은 순서대로 상위 5개 뽑아줘", engine, client)
if isinstance(result_df1, pd.DataFrame):
    display(result_df1)


[🗣️ 사용자 질문] 언론사별 기사 건수가 많은 순서대로 상위 5개 뽑아줘
[🤖 AI 작성 SQL]
SELECT press, COUNT(*) AS article_count
FROM articles
WHERE press IS NOT NULL
GROUP BY press
ORDER BY article_count DESC
LIMIT 5;
✅ 조회 성공: 총 5건 반환


,press,article_count
0,OSEN,49
1,뉴시스,29
2,스타뉴스,27
3,스포츠조선,27
4,마이데일리,24


### 🛠️ 테스트 2: 자연어 테이블 생성 (DDL / CREATE TABLE)
- 사용자의 자연어 요청: `"관심 기사를 스크랩해서 메모할 bookmark_articles 테이블을 만들어줘 (id 기본키 자동증가, article_id 정수형, memo 텍스트, created_at 현재시간)"`
- AI가 `CREATE TABLE IF NOT EXISTS` 작성 ➔ `with engine.begin()`으로 트랜잭션 반영

In [ ]:
# 테스트 2 실행: DDL 테이블 생성
ddl_result = ask_ai_db(
    "관심 기사를 스크랩해서 메모할 bookmark_articles 테이블을 만들어줘 (id 기본키 자동증가, article_id 정수형, memo 텍스트, created_at 현재시간)",
    engine,
    client
)

# inspect로 실제 테이블이 생성되었는지 확인
inspector = inspect(engine)
print("\n[현재 데이터베이스 테이블 목록]", inspector.get_table_names())


[🗣️ 사용자 질문] 관심 기사를 스크랩해서 메모할 bookmark_articles 테이블을 만들어줘 (id 기본키 자동증가, article_id 정수형, memo 텍스트, created_at 현재시간)
[🤖 AI 작성 SQL]
CREATE TABLE IF NOT EXISTS bookmark_articles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    article_id INTEGER,
    memo TEXT,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);
✅ DDL 실행 완료 (영향받은 행: -1건)

[현재 데이터베이스 테이블 목록] ['ai_reports', 'article_keywords', 'articles', 'bookmark_articles', 'search_queries']


### 📝 테스트 3: 자연어 데이터 등록 / 조회 / 수정 (CRUD DML)
1. **데이터 등록 (Create)**: `"bookmark_articles 테이블에 article_id=1, memo='2026 개막전 필독 기사'로 데이터 하나 등록해줘"`
2. **데이터 조회 (Read)**: `"방금 등록한 북마크 목록 전체 보여줘"`
3. **데이터 수정 (Update)**: `"article_id=1인 북마크 메모를 '완독 완료'로 변경해줘"`

In [ ]:
# 3-1. 데이터 등록 (INSERT)
ask_ai_db("bookmark_articles 테이블에 article_id=1, memo='2026 개막전 필독 기사'로 데이터 하나 등록해줘", engine, client)

# 3-2. 데이터 조회 (SELECT)
df_bookmark = ask_ai_db("bookmark_articles 테이블의 모든 데이터 목록 보여줘", engine, client)
if isinstance(df_bookmark, pd.DataFrame):
    display(df_bookmark)

# 3-3. 데이터 수정 (UPDATE)
ask_ai_db("bookmark_articles 테이블에서 article_id=1인 행의 memo를 '완독 완료'로 바꿔줘", engine, client)

# 수정 결과 재확인
df_updated = ask_ai_db("bookmark_articles 테이블의 모든 데이터 목록 보여줘", engine, client)
if isinstance(df_updated, pd.DataFrame):
    display(df_updated)


[🗣️ 사용자 질문] bookmark_articles 테이블에 article_id=1, memo='2026 개막전 필독 기사'로 데이터 하나 등록해줘
[🤖 AI 작성 SQL]
INSERT INTO bookmark_articles (article_id, memo) VALUES (1, '2026 개막전 필독 기사');
✅ WRITE 실행 완료 (영향받은 행: 1건)

[🗣️ 사용자 질문] bookmark_articles 테이블의 모든 데이터 목록 보여줘
[🤖 AI 작성 SQL]
SELECT * FROM bookmark_articles;
✅ 조회 성공: 총 1건 반환


,id,article_id,memo,created_at
0,1,1,2026 개막전 필독 기사,2026-09-11 05:39:32



[🗣️ 사용자 질문] bookmark_articles 테이블에서 article_id=1인 행의 memo를 '완독 완료'로 바꿔줘
[🤖 AI 작성 SQL]
UPDATE bookmark_articles
SET memo = '완독 완료'
WHERE article_id = 1;
✅ WRITE 실행 완료 (영향받은 행: 1건)

[🗣️ 사용자 질문] bookmark_articles 테이블의 모든 데이터 목록 보여줘
[🤖 AI 작성 SQL]
SELECT * FROM bookmark_articles;
✅ 조회 성공: 총 1건 반환


,id,article_id,memo,created_at
0,1,1,완독 완료,2026-09-11 05:39:32


In [ ]:
ask_ai_db("기사들 중 날짜 데이터 형식이 잘못 된 걸 조회해줘",engine,client)


[🗣️ 사용자 질문] 기사들 중 날짜 데이터 형식이 잘못 된 걸 조회해줘
[🤖 AI 작성 SQL]
SELECT *
FROM articles
WHERE date IS NULL
   OR trim(date) = ''
   OR length(date) <> 10
   OR date NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
   OR date(date, '+0 days') IS NULL
   OR date(date, '+0 days') <> date
ORDER BY id;
✅ 조회 성공: 총 0건 반환


,id,category,title,press,date,url,content,created_at


### 🔄 테스트 4: 자가 수정(Self-Correction) 복구 능력 검증
- 존재하지 않는 가상의 컬럼명을 질의하도록 유도하여 `SQLAlchemyError`를 발생시키고,
- AI가 오류 내용을 분석하여 올바른 컬럼으로 자동 보정하여 성공하는지 확인합니다.

In [ ]:
# 존재하지 않는 컬럼명(reporter_name) 질의 유도 ➔ 자가 수정 동작 확인
df_self_correct = ask_ai_db("articles 테이블에서 기자이름(reporter_name)과 기사제목을 3건만 보여줘", engine, client)
if isinstance(df_self_correct, pd.DataFrame):
    display(df_self_correct)


[🗣️ 사용자 질문] articles 테이블에서 기자이름(reporter_name)과 기사제목을 3건만 보여줘
[🤖 AI 작성 SQL]
SELECT NULL AS reporter_name, title
FROM articles
LIMIT 3;
✅ 조회 성공: 총 3건 반환


,reporter_name,title
0,None,"‘사령탑 200승’ SSG 이숭용 감독 “데뷔 첫 선발승 거둔 이로운, 기쁨..."
1,None,“후반기부터 랜더스 야구”…이숭용 감독의 통산 200승의 영광을 선수...
2,None,"'통산 200승 고지' 이숭용 감독은 선수들에게 공 돌렸다…""아빌라 오면..."


### 🗑️ 테스트 5: 제목에 특정 문구가 포함된 데이터 조건부 조회 및 삭제 (SELECT / DELETE)
데이터베이스 관리 시 가장 주의해야 할 작업은 **조건부 데이터 삭제**입니다.
실수나 오작동으로 인한 데이터 유실을 방지하기 위해 **(추가 및 조회)** 단계와 **(조건부 삭제 및 검증)** 단계를 2개의 독립된 코드 셀로 나누어 실습합니다.

- **1. [코드셀 1] 추가 + 조회**: `url` 필수값(NOT NULL)을 포함한 테스트 기사 2건 등록 ➔ 제목에 특정 문구가 들어간 기사 조회
- **2. [코드셀 2] 삭제 + 검증**: 특정 문구가 포함된 기사 조건부 삭제 ➔ 잔여 데이터 확인 ➔ 무차별 전체 삭제 차단 가드 검증


In [29]:
# [테스트 5-A] 1단계: 테스트용 샘플 기사 등록 (url 필수 컬럼 포함) + 2단계: 특정 문구 기사 조회

# 5-1. 안전한 실습을 위한 테스트용 샘플 기사 등록 (INSERT)
print("=== 1단계: 테스트용 샘플 데이터 등록 (url 필수값 포함) ===")
ask_ai_db(
    "articles 테이블에 title='[임시테스트] 2026 KBO 시범경기 일정 안내', press='테스트일보', date='2026-09-11', url='https://test.news/temp01' 데이터 추가해줘",
    engine, client
)
ask_ai_db(
    "articles 테이블에 title='[임시테스트] 우천 취소 경기 재편성 알림', press='테스트스포츠', date='2026-09-11', url='https://test.news/temp02' 데이터 추가해줘",
    engine, client
)

# 5-2. 제목에 특정 문구('[임시테스트]')가 포함된 기사 조회 (SELECT)
print("\n=== 2단계: 제목에 특정 문구가 들어간 기사 조회 ===")
df_target = ask_ai_db("articles 테이블에서 제목에 '[임시테스트]'가 들어간 기사를 조회해줘", engine, client)
if isinstance(df_target, pd.DataFrame):
    display(df_target)


=== 1단계: 테스트용 샘플 데이터 등록 (url 필수값 포함) ===

[🗣️ 사용자 질문] articles 테이블에 title='[임시테스트] 2026 KBO 시범경기 일정 안내', press='테스트일보', date='2026-09-11', url='https://test.news/temp01' 데이터 추가해줘
[🤖 AI 작성 SQL]
INSERT INTO articles (title, press, date, url) VALUES ('[임시테스트] 2026 KBO 시범경기 일정 안내', '테스트일보', '2026-09-11', 'https://test.news/temp01');
✅ WRITE 실행 완료 (영향받은 행: 1건)

[🗣️ 사용자 질문] articles 테이블에 title='[임시테스트] 우천 취소 경기 재편성 알림', press='테스트스포츠', date='2026-09-11', url='https://test.news/temp02' 데이터 추가해줘
[🤖 AI 작성 SQL]
INSERT INTO articles (title, press, date, url) VALUES ('[임시테스트] 우천 취소 경기 재편성 알림', '테스트스포츠', '2026-09-11', 'https://test.news/temp02');
✅ WRITE 실행 완료 (영향받은 행: 1건)

=== 2단계: 제목에 특정 문구가 들어간 기사 조회 ===

[🗣️ 사용자 질문] articles 테이블에서 제목에 '[임시테스트]'가 들어간 기사를 조회해줘
[🤖 AI 작성 SQL]
SELECT *
FROM articles
WHERE title LIKE '%[임시테스트]%';
✅ 조회 성공: 총 2건 반환


,id,category,title,press,date,url,content,created_at
0,668,None,[임시테스트] 2026 KBO 시범경기 일정 안내,테스트일보,2026-09-11,https://test.news/temp01,None,2026-09-11 06:46:38
1,669,None,[임시테스트] 우천 취소 경기 재편성 알림,테스트스포츠,2026-09-11,https://test.news/temp02,None,2026-09-11 06:46:40


In [30]:
# [테스트 5-B] 3단계: 특정 문구 기사 조건부 삭제 + 4단계: 삭제 검증 + 5단계: 보안 가드 확인

# 5-3. 제목에 특정 문구('[임시테스트]')가 포함된 기사 조건부 삭제 (DELETE)
print("=== 3단계: 제목에 특정 문구가 들어간 기사 안전 삭제 ===")
delete_res = ask_ai_db("articles 테이블에서 제목에 '[임시테스트]'가 포함된 기사를 삭제해줘", engine, client)

# 5-4. 삭제 결과 재확인 (조회 결과 0건 확인)
print("\n=== 4단계: 삭제 후 잔여 데이터 검증 ===")
df_after = ask_ai_db("articles 테이블에서 제목에 '[임시테스트]'가 들어간 기사를 다시 조회해줘", engine, client)
if isinstance(df_after, pd.DataFrame):
    display(df_after)

# 5-5. [보안 가드 검증] WHERE 조건 없는 무차별 전체 삭제 요청 사전 차단 테스트
print("\n=== 5단계: 보안 가드(무차별 DELETE 차단) 검증 ===")
blocked_res = ask_ai_db("articles 테이블의 데이터를 전부 다 삭제해줘", engine, client)


=== 3단계: 제목에 특정 문구가 들어간 기사 안전 삭제 ===

[🗣️ 사용자 질문] articles 테이블에서 제목에 '[임시테스트]'가 포함된 기사를 삭제해줘
[🤖 AI 작성 SQL]
DELETE FROM articles WHERE title LIKE '%[임시테스트]%';
✅ WRITE 실행 완료 (영향받은 행: 2건)

=== 4단계: 삭제 후 잔여 데이터 검증 ===

[🗣️ 사용자 질문] articles 테이블에서 제목에 '[임시테스트]'가 들어간 기사를 다시 조회해줘
[🤖 AI 작성 SQL]
SELECT *
FROM articles
WHERE title LIKE '%[임시테스트]%';
✅ 조회 성공: 총 0건 반환


,id,category,title,press,date,url,content,created_at



=== 5단계: 보안 가드(무차별 DELETE 차단) 검증 ===

[🗣️ 사용자 질문] articles 테이블의 데이터를 전부 다 삭제해줘
[🤖 AI 작성 SQL]
DELETE FROM articles;
⚠️ 쿼리 안전성 검사 차단: WHERE 조건절이 없는 전체 행 DELETE 구문은 실행할 수 없습니다.


### 🔗 테스트 6: 대화 맥락 기억(Context) 기반 연속 작업 실습
**'어떤 조건으로 조회한 후 ➔ 조건을 반복하지 않고 "방금 조회된 기사들을 삭제/북마크해줘"'**라고 명령했을 때,
AI가 직전 대화에서 조회된 기사 ID 목록을 기억하여 정밀하게 연속 쿼리(`WHERE id IN (...)`)를 작성하고 실행하는 워크플로우를 실습합니다.

- **1. [코드셀 1] 제작 + 조회**: 테스트 기사 2건 등록 ➔ 특정 문구 기사 조회 (ID 목록 메모리 자동 보관)
- **2. [코드셀 2] 삭제 + 검증**: 조건을 다시 말하지 않고 `"방금 조회된 기사들을 삭제해줘"` 실행 ➔ 2건 완벽 삭제 확인


In [37]:
# [테스트 6-A] 1단계: 테스트용 기사 2건 등록 + 2단계: 특정 문구 기사 조회 (컨텍스트 메모리 보관)
from pathlib import Path
from mini_project_0909.ai_db_service import SQLAlchemyAIDatabaseEngine

# 컨텍스트 지원 AI DB 엔진 초기화 (baseball_news.db 대상)
ai_engine = SQLAlchemyAIDatabaseEngine(db_path=Path("baseball_news.db"), client=client)

# 6-1. 안전한 실습을 위한 테스트용 샘플 기사 2건 등록
print("=== 1단계: 테스트용 샘플 기사 2건 등록 ===")
ai_engine.ask("articles 테이블에 title='[연속테스트] 1호 기사', press='A신문', date='2026-09-11', url='https://test.news/chain-01' 데이터 추가해줘")
ai_engine.ask("articles 테이블에 title='[연속테스트] 2호 기사', press='B신문', date='2026-09-11', url='https://test.news/chain-02' 데이터 추가해줘")

# 6-2. 특정 문구('[연속테스트]') 기사 조회 (컨텍스트 메모리 자동 보관)
print("\n=== 2단계: 특정 문구 기사 조회 (컨텍스트 기억) ===")
res_read = ai_engine.ask("articles 테이블에서 제목에 '[연속테스트]'가 들어간 기사 조회해줘")
print(f"[🤖 AI 작성 SQL]\n{res_read.get('sql')}")
print(f"✅ {res_read.get('message')}")
if res_read.get("data"):
    display(pd.DataFrame(res_read["data"]))


=== 1단계: 테스트용 샘플 기사 2건 등록 ===

=== 2단계: 특정 문구 기사 조회 (컨텍스트 기억) ===
[🤖 AI 작성 SQL]
SELECT * FROM articles WHERE title LIKE '%[연속테스트]%';
✅ 데이터베이스에서 총 2건의 데이터를 조회했습니다.


,id,title,press,date,url
0,None,[연속테스트] 1호 기사,A신문,2026-09-11,https://test.news/chain-01
1,None,[연속테스트] 2호 기사,B신문,2026-09-11,https://test.news/chain-02


In [36]:
# [테스트 6-B] 3단계: [핵심] 직전 컨텍스트 기반 삭제 + 4단계: 삭제 검증

# 6-3. 직전 컨텍스트 기반 삭제: 조건을 반복하지 않고 '방금 조회된 기사들 삭제' 명령
print("=== 3단계: [핵심] 방금 조회된 기사들을 삭제해줘 ===")
res_del = ai_engine.ask("방금 조회된 기사들을 삭제해줘")
print(f"[🤖 AI 작성 SQL]\n{res_del.get('sql')}")
print(f"✅ {res_del.get('message')}")

# 6-4. 삭제 결과 재검증 (0건 확인)
print("\n=== 4단계: 삭제 후 잔여 데이터 검증 ===")
res_after = ai_engine.ask("articles 테이블에서 제목에 '[연속테스트]'가 들어간 기사 다시 조회해줘")
print(f"[🤖 AI 작성 SQL]\n{res_after.get('sql')}")
print(f"✅ {res_after.get('message')}")
if res_after.get("data"):
    display(pd.DataFrame(res_after["data"]))


=== 3단계: [핵심] 방금 조회된 기사들을 삭제해줘 ===
[🤖 AI 작성 SQL]
DELETE FROM articles WHERE title LIKE '%[연속테스트]%';
✅ 데이터베이스에서 조건에 일치하는 데이터 총 2건을 성공적으로 삭제했습니다.

=== 4단계: 삭제 후 잔여 데이터 검증 ===
[🤖 AI 작성 SQL]
SELECT * FROM articles WHERE title LIKE '%[연속테스트]%';
✅ 조건에 부합하는 데이터가 데이터베이스에 존재하지 않습니다.


## 🌤️ 7. KBO 전국 11개 구장 특정 시간대 날씨 기록 DB 적재 및 다채널 조회 실습

기상청 단기예보(초단기실황 API) 모듈(`weather.py`)의 데이터를 바탕으로,
1. **기상 이력 테이블(`stadium_weather_history`) 스키마 정의 및 초기화**
2. **KBO 전국 11개 구장의 실시간 관측 데이터 수집 및 고속 UPSERT(중복 방지) 적재**
3. **주요 야구 시간대(11:00 사전점검, 14:00 낮경기, 18:00 야간경기)별 날씨 이력 누적 및 시점별 조회**
4. **AI DB 엔진(`SQLAlchemyAIDatabaseEngine`)을 활용한 자연어 기반 기상 데이터 질의 및 통계 분석**

전 과정을 단계별 독립 실행 셀로 실습합니다.

In [38]:
# [테스트 7-A] 1단계: 기상 데이터 이력 테이블(stadium_weather_history) DDL 정의 및 초기화
import sqlite3
from pathlib import Path
import pandas as pd

db_path = Path("baseball_news.db")

# 1. stadium_weather_history DDL 스키마
WEATHER_HISTORY_DDL = """
CREATE TABLE IF NOT EXISTS stadium_weather_history (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    stadium_id      TEXT NOT NULL,          -- 'jamsil', 'gocheok', 'munhak' 등
    stadium_name    TEXT NOT NULL,          -- '서울 잠실야구장' 등
    base_date       TEXT NOT NULL,          -- 'YYYY-MM-DD'
    base_time       TEXT NOT NULL,          -- 'HH:00' (예: '11:00', '14:00', '18:00')
    temp            REAL,                   -- 기온 수치 (℃)
    temp_str        TEXT,                   -- 기온 문자열 ('24.5℃')
    rain            REAL DEFAULT 0.0,       -- 1시간 강수량 (mm)
    humidity        REAL,                   -- 습도 (%)
    wind_speed      REAL DEFAULT 0.0,       -- 풍속 (m/s)
    pty             INTEGER DEFAULT 0,      -- 강수 형태 (0:없음, 1:비, 2:비/눈, 3:눈, 5:빗방울 등)
    status_label    TEXT,                   -- '🟢 정상 진행 가능', '🔴 우천 취소 우려' 등
    badge_class     TEXT,                   -- 'badge-safe', 'badge-danger' 등
    status_desc     TEXT,                   -- 경기 상태 설명
    icon            TEXT,                   -- '☀️', '🌧️', '⛈️' 등
    created_at      DATETIME DEFAULT CURRENT_TIMESTAMP,
    UNIQUE (stadium_id, base_date, base_time)
);

CREATE INDEX IF NOT EXISTS idx_swh_date_time ON stadium_weather_history(base_date, base_time);
CREATE INDEX IF NOT EXISTS idx_swh_stadium ON stadium_weather_history(stadium_id);
"""

conn = sqlite3.connect(db_path)
try:
    cursor = conn.cursor()
    cursor.executescript(WEATHER_HISTORY_DDL)
    conn.commit()
    print("✅ stadium_weather_history 테이블 및 인덱스 생성 완료!")
    
    # 스키마 컬럼 정보 확인
    cursor.execute("PRAGMA table_info(stadium_weather_history);")
    schema_info = cursor.fetchall()
    df_schema = pd.DataFrame(schema_info, columns=["cid", "name", "type", "notnull", "dflt_value", "pk"])
    display(df_schema[["cid", "name", "type", "notnull", "pk"]])
finally:
    conn.close()

✅ stadium_weather_history 테이블 및 인덱스 생성 완료!


,cid,name,type,notnull,pk
0,0,id,INTEGER,0,1
1,1,stadium_id,TEXT,1,0
2,2,stadium_name,TEXT,1,0
3,3,base_date,TEXT,1,0
4,4,base_time,TEXT,1,0
5,5,temp,REAL,0,0
6,6,temp_str,TEXT,0,0
7,7,rain,REAL,0,0
8,8,humidity,REAL,0,0
9,9,wind_speed,REAL,0,0


In [39]:
# [테스트 7-B] 2단계: 기상청 API 연동 11개 구장 실시간 날씨 데이터 수집 및 고속 적재 (UPSERT)
import os
from datetime import datetime
from dotenv import load_dotenv
from mini_project_0909.weather import STADIUMS, fetch_stadium_weather, get_kma_base_datetime

load_dotenv(override=True)

def save_stadium_weather_to_db(db_path: Path = Path("baseball_news.db"), custom_date: str = None, custom_time: str = None):
    """
    KBO 11개 구장 날씨를 수집하여 stadium_weather_history 테이블에 고속 UPSERT 적재합니다.
    """
    api_key = os.getenv("DATA_GO_KR_API_KEY", "")
    if not api_key:
        print("⚠️ DATA_GO_KR_API_KEY 환경변수가 설정되지 않았습니다.")
        return []

    # 기준 일자 및 시각 결정
    if custom_date and custom_time:
        base_date_api = custom_date.replace("-", "")
        base_time_api = custom_time.replace(":", "")
        display_date = custom_date
        display_time = custom_time
    else:
        base_date_api, base_time_api = get_kma_base_datetime()
        display_date = f"{base_date_api[:4]}-{base_date_api[4:6]}-{base_date_api[6:]}"
        display_time = f"{base_time_api[:2]}:{base_time_api[2:]}"

    print(f"📡 기상청 초단기실황 데이터 수집 중... (기준시각: {display_date} {display_time})")

    conn = sqlite3.connect(db_path)
    try:
        cursor = conn.cursor()
        upsert_sql = """
        INSERT INTO stadium_weather_history (
            stadium_id, stadium_name, base_date, base_time,
            temp, temp_str, rain, humidity, wind_speed, pty,
            status_label, badge_class, status_desc, icon
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(stadium_id, base_date, base_time) DO UPDATE SET
            temp = excluded.temp,
            temp_str = excluded.temp_str,
            rain = excluded.rain,
            humidity = excluded.humidity,
            wind_speed = excluded.wind_speed,
            pty = excluded.pty,
            status_label = excluded.status_label,
            badge_class = excluded.badge_class,
            status_desc = excluded.status_desc,
            icon = excluded.icon,
            created_at = CURRENT_TIMESTAMP;
        """

        records = []
        for std in STADIUMS:
            w = fetch_stadium_weather(std, api_key, base_date_api, base_time_api)
            # 수치형 기온 파싱
            temp_val = None
            if w.get("temp") and w["temp"] != "--":
                try:
                    temp_val = float(w["temp"].replace("℃", "").strip())
                except ValueError:
                    temp_val = None

            # 수치형 강수량 파싱
            rain_val = 0.0
            if w.get("rain"):
                try:
                    rain_val = float(w["rain"].replace("mm", "").strip())
                except ValueError:
                    rain_val = 0.0

            # 수치형 습도 파싱
            humidity_val = None
            if w.get("humidity") and w["humidity"] != "--":
                try:
                    humidity_val = float(w["humidity"].replace("%", "").strip())
                except ValueError:
                    humidity_val = None

            # 수치형 풍속 파싱
            wind_val = 0.0
            if w.get("wind_speed"):
                try:
                    wind_val = float(w["wind_speed"].replace("m/s", "").strip())
                except ValueError:
                    wind_val = 0.0

            cursor.execute(
                upsert_sql,
                (
                    std["id"],
                    std["name"],
                    display_date,
                    display_time,
                    temp_val,
                    w.get("temp", "--"),
                    rain_val,
                    humidity_val,
                    wind_val,
                    int(w.get("pty", 0) or 0),
                    w.get("status_label", "🟢 정상 진행 가능"),
                    w.get("badge_class", "badge-safe"),
                    w.get("status_desc", ""),
                    w.get("icon", "☀️"),
                ),
            )
            records.append({
                "구장명": std["name"],
                "기준일시": f"{display_date} {display_time}",
                "날씨": w.get("icon", "☀️"),
                "기온": w.get("temp", "--"),
                "강수량": f"{rain_val} mm",
                "습도": f"{humidity_val}%" if humidity_val is not None else "--",
                "풍속": f"{wind_val} m/s",
                "경기 진행 상태": w.get("status_label", "🟢 정상 진행 가능")
            })

        conn.commit()
        print(f"✅ 총 {len(records)}개 구장 날씨 기록 적재 완료 (중복 방지 UPSERT 적용)")
        return records
    finally:
        conn.close()

# 실제 기상청 API 호출 및 DB 적재 실행
saved_records = save_stadium_weather_to_db()
if saved_records:
    df_current_weather = pd.DataFrame(saved_records)
    display(df_current_weather)


📡 기상청 초단기실황 데이터 수집 중... (기준시각: 2026-09-11 15:00)
✅ 총 11개 구장 날씨 기록 적재 완료 (중복 방지 UPSERT 적용)


,구장명,기준일시,날씨,기온,강수량,습도,풍속,경기 진행 상태
0,서울 잠실야구장,2026-09-11 15:00,☀️,25.5℃,0.0 mm,42.0%,1.9 m/s,🟢 정상 진행 가능
1,서울 고척스카이돔,2026-09-11 15:00,☀️,27℃,0.0 mm,33.0%,1.9 m/s,🔵 돔구장
2,인천 SSG랜더스필드,2026-09-11 15:00,☀️,26.1℃,0.0 mm,39.0%,1.9 m/s,🟢 정상 진행 가능
3,수원 KT위즈파크,2026-09-11 15:00,☀️,25.7℃,0.0 mm,37.0%,3.1 m/s,🟢 정상 진행 가능
4,대전 한화생명이글스파크,2026-09-11 15:00,☀️,25.9℃,0.0 mm,42.0%,1.7 m/s,🟢 정상 진행 가능
5,대구 삼성라이온즈파크,2026-09-11 15:00,☀️,27.6℃,0.0 mm,38.0%,0.8 m/s,🟢 정상 진행 가능
6,광주-기아 챔피언스필드,2026-09-11 15:00,☀️,28.5℃,0.0 mm,45.0%,1.3 m/s,🟢 정상 진행 가능
7,부산 사직야구장,2026-09-11 15:00,☀️,25.3℃,0.0 mm,59.0%,1.4 m/s,🟢 정상 진행 가능
8,창원 NC파크,2026-09-11 15:00,☀️,30℃,0.0 mm,37.0%,1.1 m/s,🟢 정상 진행 가능
9,포항야구장 (제2구장),2026-09-11 15:00,☀️,24.8℃,0.0 mm,62.0%,3.0 m/s,🟢 정상 진행 가능


In [40]:
# [테스트 7-C] 3단계: 야구 주요 시간대(11:00 점검, 14:00 낮경기, 18:00 야간경기) 이력 누적 및 시점별 조회 함수
from datetime import datetime, timedelta

def populate_sample_baseball_time_weather(db_path: Path = Path("baseball_news.db")):
    """
    풍부한 시간대별 이력 테스트를 위해,
    - 오늘 11:00 (사전점검)
    - 어제 18:00 (어제 야간경기 - 광주/대전 우천 상황 포함)
    시뮬레이션 관측 데이터를 DB에 추가 적재합니다.
    """
    conn = sqlite3.connect(db_path)
    try:
        cursor = conn.cursor()
        upsert_sql = """
        INSERT INTO stadium_weather_history (
            stadium_id, stadium_name, base_date, base_time,
            temp, temp_str, rain, humidity, wind_speed, pty,
            status_label, badge_class, status_desc, icon
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(stadium_id, base_date, base_time) DO UPDATE SET
            temp = excluded.temp,
            temp_str = excluded.temp_str,
            rain = excluded.rain,
            humidity = excluded.humidity,
            wind_speed = excluded.wind_speed,
            pty = excluded.pty,
            status_label = excluded.status_label,
            badge_class = excluded.badge_class,
            status_desc = excluded.status_desc,
            icon = excluded.icon,
            created_at = CURRENT_TIMESTAMP;
        """
        today_str = datetime.now().strftime("%Y-%m-%d")
        yesterday_str = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

        # 1. 오늘 11:00 사전 점검 데이터 (맑음/구름 상태)
        for std in STADIUMS:
            cursor.execute(upsert_sql, (
                std["id"], std["name"], today_str, "11:00",
                21.5, "21.5℃", 0.0, 60.0, 2.1, 0,
                "🟢 정상 진행 가능", "badge-safe", "경기 진행에 특이사항이 없습니다.", "☀️"
            ))

        # 2. 어제 18:00 야간경기 데이터 (광주 폭우, 대전 우천 주의 시뮬레이션)
        for std in STADIUMS:
            if std["id"] == "gwangju":
                cursor.execute(upsert_sql, (
                    std["id"], std["name"], yesterday_str, "18:00",
                    19.0, "19.0℃", 6.5, 95.0, 5.2, 1,
                    "🔴 우천 취소 우려", "badge-danger", "시간당 6.5mm의 많은 비로 우천 취소 가능성이 높습니다.", "⛈️"
                ))
            elif std["id"] == "daejeon":
                cursor.execute(upsert_sql, (
                    std["id"], std["name"], yesterday_str, "18:00",
                    20.5, "20.5℃", 1.8, 85.0, 3.5, 1,
                    "🟡 우천 주의 (방수포)", "badge-caution", "시간당 1.8mm의 비로 방수포 설치 및 경기 지연 가능성", "🌧️"
                ))
            else:
                cursor.execute(upsert_sql, (
                    std["id"], std["name"], yesterday_str, "18:00",
                    23.0, "23.0℃", 0.0, 55.0, 1.8, 0,
                    "🟢 정상 진행 가능", "badge-safe", "쾌적한 야간경기 진행 가능", "☀️"
                ))

        conn.commit()
        print("✅ 어제 18:00 야간경기 및 오늘 11:00 사전점검 시뮬레이션 데이터 적재 완료!")
    finally:
        conn.close()

def query_stadium_weather_by_time(base_date: str, base_time: str, db_path: Path = Path("baseball_news.db")):
    """지정된 날짜 및 시간대의 11개 구장 날씨 조회"""
    conn = sqlite3.connect(db_path)
    try:
        query = """
        SELECT stadium_name, base_date, base_time, temp_str, rain, humidity, wind_speed, status_label, icon
        FROM stadium_weather_history
        WHERE base_date = ? AND base_time = ?
        ORDER BY id;
        """
        df = pd.read_sql_query(query, conn, params=(base_date, base_time))
        return df
    finally:
        conn.close()

# 시뮬레이션 데이터 적재 실행
populate_sample_baseball_time_weather()

# 어제 18:00 야간경기 시점 조회
yesterday_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"\n📋 [조회 결과] {yesterday_date} 18:00 야간경기 11개 구장 날씨:")
df_yesterday_18 = query_stadium_weather_by_time(yesterday_date, "18:00")
display(df_yesterday_18)


✅ 어제 18:00 야간경기 및 오늘 11:00 사전점검 시뮬레이션 데이터 적재 완료!

📋 [조회 결과] 2026-09-10 18:00 야간경기 11개 구장 날씨:


,stadium_name,base_date,base_time,temp_str,rain,humidity,wind_speed,status_label,icon
0,광주-기아 챔피언스필드,2026-09-10,18:00,19.0℃,6.5,95.0,5.2,🔴 우천 취소 우려,⛈️
1,서울 잠실야구장,2026-09-10,18:00,23.0℃,0.0,55.0,1.8,🟢 정상 진행 가능,☀️
2,서울 고척스카이돔,2026-09-10,18:00,23.0℃,0.0,55.0,1.8,🟢 정상 진행 가능,☀️
3,인천 SSG랜더스필드,2026-09-10,18:00,23.0℃,0.0,55.0,1.8,🟢 정상 진행 가능,☀️
4,수원 KT위즈파크,2026-09-10,18:00,23.0℃,0.0,55.0,1.8,🟢 정상 진행 가능,☀️
5,대전 한화생명이글스파크,2026-09-10,18:00,20.5℃,1.8,85.0,3.5,🟡 우천 주의 (방수포),🌧️
6,대구 삼성라이온즈파크,2026-09-10,18:00,23.0℃,0.0,55.0,1.8,🟢 정상 진행 가능,☀️
7,부산 사직야구장,2026-09-10,18:00,23.0℃,0.0,55.0,1.8,🟢 정상 진행 가능,☀️
8,창원 NC파크,2026-09-10,18:00,23.0℃,0.0,55.0,1.8,🟢 정상 진행 가능,☀️
9,포항야구장 (제2구장),2026-09-10,18:00,23.0℃,0.0,55.0,1.8,🟢 정상 진행 가능,☀️


In [41]:
# [테스트 7-D] 4단계: AI DB 제어 엔진을 통한 자연어 날씨 질의 및 통계 분석 실습
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from mini_project_0909.ai_db_service import SQLAlchemyAIDatabaseEngine

load_dotenv(override=True)

# OpenAI 클라이언트 초기화 (기존 client 재활용 또는 신규 생성)
try:
    _test_client = client
except NameError:
    _test_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# SQLAlchemy 2.0 기반 AI DB 엔진 초기화 (stadium_weather_history 자동 리플렉션)
ai_weather_engine = SQLAlchemyAIDatabaseEngine(
    db_path=Path("baseball_news.db"),
    client=_test_client,
    model="gpt-5.6-luna"
)

print("🤖 [AI DB 엔진] stadium_weather_history 테이블 인식 완료!")

# 1. 특정 일자/시간 전체 구장 날씨 자연어 질의
print("\n=== 질의 1: 특정 일자/시간 구장 날씨 조회 ===")
res1 = ai_weather_engine.ask("stadium_weather_history 테이블에서 오늘 수집된 구장 날씨 전체 조회해줘")
print(f"[🤖 AI 작성 SQL]\n{res1.get('sql')}\n")
print(f"✅ {res1.get('message')}")
if res1.get("data"):
    display(pd.DataFrame(res1["data"]))

# 2. 우천 주의 / 취소 우려 구장 통계 질의
print("\n=== 질의 2: 비가 오거나 경기 진행에 주의가 필요한 구장 질의 ===")
res2 = ai_weather_engine.ask("stadium_weather_history 테이블에서 비가 오거나(rain > 0) 우천 주의/취소 우려인 구장을 조회해줘")
print(f"[🤖 AI 작성 SQL]\n{res2.get('sql')}\n")
print(f"✅ {res2.get('message')}")
if res2.get("data"):
    display(pd.DataFrame(res2["data"]))

# 3. 구장별 기온 순위 통계 질의
print("\n=== 질의 3: 기온이 가장 높은 상위 5개 구장 순위 ===")
res3 = ai_weather_engine.ask("stadium_weather_history 테이블에서 기온(temp)이 가장 높은 구장 5곳을 기온 내림차순으로 정렬해서 구장명과 기온만 보여줘")
print(f"[🤖 AI 작성 SQL]\n{res3.get('sql')}\n")
print(f"✅ {res3.get('message')}")
if res3.get("data"):
    display(pd.DataFrame(res3["data"]))


🤖 [AI DB 엔진] stadium_weather_history 테이블 인식 완료!

=== 질의 1: 특정 일자/시간 구장 날씨 조회 ===
[🤖 AI 작성 SQL]
SELECT * FROM stadium_weather_history WHERE base_date = date('now','localtime');

✅ 데이터베이스에서 총 22건의 데이터를 조회했습니다.


,id,stadium_id,stadium_name,base_date,base_time,temp,temp_str,rain,humidity,wind_speed,pty,status_label,badge_class,status_desc,icon,created_at
0,13,jamsil,서울 잠실야구장,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
1,14,gocheok,서울 고척스카이돔,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
2,15,munhak,인천 SSG랜더스필드,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
3,16,suwon,수원 KT위즈파크,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
4,17,daejeon,대전 한화생명이글스파크,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
5,18,daegu,대구 삼성라이온즈파크,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
6,19,gwangju,광주-기아 챔피언스필드,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
7,20,sajik,부산 사직야구장,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
8,21,changwon,창원 NC파크,2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40
9,22,pohang,포항야구장 (제2구장),2026-09-11,11:00,21.5,21.5℃,0.0,60.0,2.1,0,🟢 정상 진행 가능,badge-safe,경기 진행에 특이사항이 없습니다.,☀️,2026-09-11 07:09:40



=== 질의 2: 비가 오거나 경기 진행에 주의가 필요한 구장 질의 ===
[🤖 AI 작성 SQL]
SELECT * FROM stadium_weather_history WHERE rain > 0 OR status_label LIKE '%우천%' OR status_desc LIKE '%우천%' OR status_label LIKE '%취소%' OR status_desc LIKE '%취소%';

✅ 데이터베이스에서 총 2건의 데이터를 조회했습니다.


,id,stadium_id,stadium_name,base_date,base_time,temp,temp_str,rain,humidity,wind_speed,pty,status_label,badge_class,status_desc,icon,created_at
0,1,gwangju,광주-기아 챔피언스필드,2026-09-10,18:00,19.0,19.0℃,6.5,95.0,5.2,1,🔴 우천 취소 우려,badge-danger,시간당 6.5mm의 많은 비로 우천 취소 가능성이 높습니다.,⛈️,2026-09-11 07:09:40
1,28,daejeon,대전 한화생명이글스파크,2026-09-10,18:00,20.5,20.5℃,1.8,85.0,3.5,1,🟡 우천 주의 (방수포),badge-caution,시간당 1.8mm의 비로 방수포 설치 및 경기 지연 가능성,🌧️,2026-09-11 07:09:40



=== 질의 3: 기온이 가장 높은 상위 5개 구장 순위 ===
[🤖 AI 작성 SQL]
SELECT stadium_name, MAX(temp) AS temp FROM stadium_weather_history GROUP BY stadium_id, stadium_name ORDER BY temp DESC LIMIT 5;

✅ 데이터베이스에서 총 5건의 데이터를 조회했습니다.


,stadium_name,temp
0,창원 NC파크,30.0
1,광주-기아 챔피언스필드,28.5
2,대구 삼성라이온즈파크,27.6
3,서울 고척스카이돔,27.0
4,인천 SSG랜더스필드,26.1
